# Report figures

Builds the plots, tables, sample galleries, and `report_summary.json` in `report_outputs/` from the CSVs and checkpoints produced by the other notebooks.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.chdir('/content/drive/MyDrive/lld-net-pcb-ml')
except ImportError:
    pass

from project_paths import setup, prepare_dataset

P = setup()
os.chdir(P.repo_root)
print('repo :', P.repo_root)
print('colab:', P.in_colab)

In [ ]:
try:
    import google.colab
    %pip -q install ultralytics pandas pyyaml opencv-python matplotlib seaborn
except ImportError:
    pass

In [ ]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

sns.set_theme(context='talk', style='whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
REPO_ROOT          = P.repo_root
DRIVE_DATASET_DIR  = P.drive_dataset
DRIVE_DATA_YAML    = P.drive_data_yaml
DRIVE_RUNS_DIR     = P.drive_runs
DRIVE_ABL_DIR      = P.drive_abl
REPORT_DIR         = P.report_dir
REPORT_PLOTS       = REPORT_DIR / 'plots'
REPORT_SAMPLES     = REPORT_DIR / 'samples'
REPORT_FAILURES    = REPORT_DIR / 'failures'
for d in (REPORT_DIR, REPORT_PLOTS, REPORT_SAMPLES, REPORT_FAILURES):
    d.mkdir(parents=True, exist_ok=True)

RUN_BASELINE = DRIVE_RUNS_DIR / 'yolo12_pcb_baseline'
P2_AUG_PROFILE = 'strong'
RUN_P2 = DRIVE_RUNS_DIR / f'yolo12n_p2_{P2_AUG_PROFILE}'
if not RUN_P2.exists():
    legacy = DRIVE_RUNS_DIR / 'yolo12n_p2'
    if legacy.exists():
        print(f'[warn] using legacy P2 folder: {legacy}')
        RUN_P2 = legacy

BASELINE_BEST = RUN_BASELINE / 'weights' / 'best.pt'
P2_BEST       = RUN_P2 / 'weights' / 'best.pt'

LOCAL_DATASET_DIR = P.local_dataset
LOCAL_DATA_YAML   = P.local_data_yaml

for p in (BASELINE_BEST, P2_BEST):
    print(f'[{"ok" if p.exists() else "warn"}] {p}')

In [ ]:
yml = prepare_dataset(P)
CLASSES = yml['names']
print('Classes:', CLASSES)

In [ ]:
def label_stats(split):
    img_dir = LOCAL_DATASET_DIR / 'images' / split
    lbl_dir = LOCAL_DATASET_DIR / 'labels' / split
    n_img = len(list(img_dir.glob('*')))
    cls_counts = np.zeros(len(CLASSES), dtype=int)
    sizes_norm = []
    n_box = 0
    for lp in lbl_dir.glob('*.txt'):
        for line in lp.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            c = int(parts[0]); w = float(parts[3]); h = float(parts[4])
            if 0 <= c < len(CLASSES):
                cls_counts[c] += 1
                sizes_norm.append((w, h))
                n_box += 1
    return n_img, n_box, cls_counts, (np.array(sizes_norm) if sizes_norm else np.zeros((0, 2)))

rows = []; all_sizes = {}; all_cls = {}
for split in ['train', 'val', 'test']:
    n_img, n_box, cls_counts, sizes = label_stats(split)
    rows.append({'split': split, 'images': n_img, 'boxes': n_box,
                 **{f'class_{cn}': int(cnt) for cn, cnt in zip(CLASSES, cls_counts)}})
    all_sizes[split] = sizes
    all_cls[split]   = cls_counts
stats_df = pd.DataFrame(rows)
stats_df.to_csv(REPORT_DIR / 'dataset_stats.csv', index=False)
stats_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASSES)); w = 0.27
for i, split in enumerate(['train', 'val', 'test']):
    ax.bar(x + (i - 1) * w, all_cls[split], width=w, label=split)
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=20)
ax.set_ylabel('# bounding boxes')
ax.set_title('Class distribution per split')
ax.legend(); fig.tight_layout()
fig.savefig(REPORT_PLOTS / 'class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Small-object share via sqrt(normalised box area) histogram.
all_train_sizes = all_sizes['train']
if len(all_train_sizes):
    sqrt_area = np.sqrt(all_train_sizes[:, 0] * all_train_sizes[:, 1])
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(sqrt_area, bins=40, edgecolor='black')
    ax.axvline(0.05, color='red', linestyle='--', label='small (sqrt(area) < 0.05)')
    ax.set_xlabel('sqrt(box area) (normalised)'); ax.set_ylabel('count')
    ax.set_title('Defect size distribution (train)')
    ax.legend(); fig.tight_layout()
    fig.savefig(REPORT_PLOTS / 'defect_size_hist.png', dpi=150)
    plt.show()
    small_ratio = float((sqrt_area < 0.05).mean())
    print(f'small-defect ratio (<0.05): {small_ratio:.2%}')
else:
    small_ratio = None

In [ ]:
def read_results_csv(p):
    f = p / 'results.csv'
    return pd.read_csv(f) if f.exists() else None

df_b = read_results_csv(RUN_BASELINE)
df_p = read_results_csv(RUN_P2)

metrics_to_plot = [
    ('train/box_loss',       'box loss (train)'),
    ('train/cls_loss',       'cls loss (train)'),
    ('val/box_loss',         'box loss (val)'),
    ('metrics/precision(B)', 'precision'),
    ('metrics/recall(B)',    'recall'),
    ('metrics/mAP50(B)',     'mAP@0.5'),
    ('metrics/mAP50-95(B)',  'mAP@0.5:0.95'),
]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
for ax, (col, title) in zip(axes, metrics_to_plot):
    plotted = False
    if df_b is not None and col in df_b.columns:
        ax.plot(df_b['epoch'], df_b[col], label='baseline', color='C0'); plotted = True
    if df_p is not None and col in df_p.columns:
        ax.plot(df_p['epoch'], df_p[col], label='P2', color='C1'); plotted = True
    ax.set_title(title); ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3)
    if plotted: ax.legend()
for ax in axes[len(metrics_to_plot):]:
    ax.axis('off')
fig.suptitle('Training curves: baseline vs P2', y=1.02)
fig.tight_layout()
fig.savefig(REPORT_PLOTS / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Copy confusion matrices and PR / F1 / P / R curves out of the run folders.
for run, tag in [(RUN_BASELINE, 'baseline'), (RUN_P2, 'p2')]:
    for src_name, dst_name in [
        ('confusion_matrix.png',            f'confusion_matrix_{tag}.png'),
        ('confusion_matrix_normalized.png', f'confusion_matrix_normalized_{tag}.png'),
        ('BoxPR_curve.png',                 f'PR_curve_{tag}.png'),
        ('BoxF1_curve.png',                 f'F1_curve_{tag}.png'),
        ('BoxP_curve.png',                  f'P_curve_{tag}.png'),
        ('BoxR_curve.png',                  f'R_curve_{tag}.png'),
        ('results.png',                     f'results_{tag}.png'),
    ]:
        src = run / src_name
        if src.exists():
            shutil.copy2(src, REPORT_PLOTS / dst_name)

In [ ]:
# Per-class AP / F1 are exposed by Ultralytics on metrics.box after val().
def per_class_metrics(weights):
    if not Path(weights).exists():
        return None
    m = YOLO(str(weights)).val(data=str(LOCAL_DATA_YAML), split='test',
                               imgsz=640, device=0 if __import__('torch').cuda.is_available() else 'cpu', verbose=False)
    p    = np.array(m.box.p)    if hasattr(m.box, 'p')    else None
    r    = np.array(m.box.r)    if hasattr(m.box, 'r')    else None
    ap50 = np.array(m.box.ap50) if hasattr(m.box, 'ap50') else None
    mp   = np.array(m.box.maps) if hasattr(m.box, 'maps') else None
    f1   = (2 * p * r) / (p + r + 1e-9) if (p is not None and r is not None) else None
    return {'p': p, 'r': r, 'ap50': ap50, 'map': mp, 'f1': f1}

pcm_b = per_class_metrics(BASELINE_BEST)
pcm_p = per_class_metrics(P2_BEST)

if pcm_b is not None and pcm_p is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(CLASSES)); w = 0.4
    for ax, key, ylabel, title in [
        (axes[0], 'ap50', 'AP@0.5', 'Per-class AP@0.5'),
        (axes[1], 'f1',   'F1',     'Per-class F1'),
    ]:
        ax.bar(x - w/2, pcm_b[key], w, label='baseline')
        ax.bar(x + w/2, pcm_p[key], w, label='P2')
        ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=20)
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(REPORT_PLOTS / 'per_class_metrics.png', dpi=150)
    plt.show()

    pc_table = pd.DataFrame({
        'class'         : CLASSES,
        'baseline_AP50' : pcm_b['ap50'], 'p2_AP50': pcm_p['ap50'],
        'baseline_F1'   : pcm_b['f1'],   'p2_F1'  : pcm_p['f1'],
        'baseline_mAP'  : pcm_b['map'],  'p2_mAP' : pcm_p['map'],
    })
    pc_table.to_csv(REPORT_DIR / 'per_class_metrics.csv', index=False)
    print(pc_table)
else:
    pc_table = None
    print('At least one checkpoint missing; per-class section skipped.')

In [ ]:
abl_csv = DRIVE_ABL_DIR / 'ablation_results.csv'
if abl_csv.exists():
    df_abl = pd.read_csv(abl_csv)

    def get_val(tag, col='map50'):
        sub = df_abl[df_abl['exp'] == tag]
        return float(sub[col].iloc[0]) if len(sub) else np.nan

    sigmas = [0, 10, 20, 30]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, col, title in [
        (axes[0], 'map50',    'mAP@0.5 vs sigma'),
        (axes[1], 'map50_95', 'mAP@0.5:0.95 vs sigma'),
        (axes[2], 'fnr',      'False Negative Rate vs sigma'),
    ]:
        b = [get_val('baseline_test_clean', col)] + [get_val(f'baseline_test_noise_s{s}', col) for s in sigmas[1:]]
        p = [get_val('p2_test_clean',       col)] + [get_val(f'p2_test_noise_s{s}',       col) for s in sigmas[1:]]
        ax.plot(sigmas, b, 'o-', label='baseline')
        ax.plot(sigmas, p, 's-', label='P2')
        ax.set_xlabel('Gaussian sigma'); ax.set_ylabel(col); ax.set_title(title)
        ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(REPORT_PLOTS / 'noise_robustness.png', dpi=150)
    plt.show()
else:
    df_abl = None
    print('ablation_results.csv missing - run p2_noise_ablation.ipynb first.')

In [ ]:
# Side-by-side prediction gallery: GT vs baseline vs P2.

def draw_yolo_labels(img, label_path, classes, color=(0, 255, 0)):
    h, w = img.shape[:2]
    out = img.copy()
    if not Path(label_path).exists():
        return out
    for line in Path(label_path).read_text(encoding='utf-8').splitlines():
        ps = line.strip().split()
        if len(ps) != 5:
            continue
        c, xc, yc, bw, bh = ps
        c = int(c); xc = float(xc); yc = float(yc); bw = float(bw); bh = float(bh)
        x1 = int((xc - bw / 2) * w); y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w); y2 = int((yc + bh / 2) * h)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, classes[c] if c < len(classes) else str(c),
                    (x1, max(15, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return out

def draw_predictions(img, pred_result, color=(0, 165, 255)):
    out = img.copy()
    boxes = pred_result.boxes
    if boxes is None or len(boxes) == 0:
        return out
    cls  = boxes.cls.cpu().numpy().astype(int)
    xyxy = boxes.xyxy.cpu().numpy().astype(int)
    conf = boxes.conf.cpu().numpy()
    names = pred_result.names
    for (x1, y1, x2, y2), c, cf in zip(xyxy, cls, conf):
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        cv2.putText(out, f'{names[c]} {cf:.2f}', (x1, max(15, y1 - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return out

test_imgs = sorted((LOCAL_DATASET_DIR / 'images' / 'test').glob('*'))
rng = np.random.default_rng(42)
sample_imgs = list(rng.choice(np.array(test_imgs, dtype=object),
                              size=min(8, len(test_imgs)), replace=False))

baseline_model = YOLO(str(BASELINE_BEST))
p2_best_model  = YOLO(str(P2_BEST))

for ip in sample_imgs:
    img = cv2.imread(str(ip))
    if img is None:
        continue
    gt_lbl = LOCAL_DATASET_DIR / 'labels' / 'test' / (ip.stem + '.txt')
    gt_img = draw_yolo_labels(img, gt_lbl, CLASSES, color=(0, 255, 0))

    b_pred = baseline_model.predict(source=str(ip), imgsz=640, conf=0.25, verbose=False)[0]
    p_pred = p2_best_model.predict (source=str(ip), imgsz=640, conf=0.25, verbose=False)[0]
    b_img = draw_predictions(img, b_pred, color=(0, 165, 255))
    p_img = draw_predictions(img, p_pred, color=(255, 80, 80))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, im, t in zip(axes, [gt_img, b_img, p_img], ['Ground Truth', 'Baseline', 'P2']):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t); ax.axis('off')
    fig.suptitle(ip.name)
    fig.tight_layout()
    fig.savefig(REPORT_SAMPLES / f'{ip.stem}_compare.png', dpi=120)
    plt.close(fig)
print('Sample comparisons saved to:', REPORT_SAMPLES)

In [ ]:
# Failure-case gallery: pick test images with the largest baseline FN+FP and show P2 alongside.

def iou_xyxy(a, b):
    inter_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0

def yolo_to_xyxy(line, w, h):
    c, xc, yc, bw, bh = line.strip().split()
    c = int(c); xc = float(xc); yc = float(yc); bw = float(bw); bh = float(bh)
    return c, [(xc - bw/2) * w, (yc - bh/2) * h, (xc + bw/2) * w, (yc + bh/2) * h]

def collect_failures(model, classes, max_cases=8, iou_thr=0.5, conf_thr=0.25):
    items = []
    for ip in test_imgs:
        lbl = LOCAL_DATASET_DIR / 'labels' / 'test' / (ip.stem + '.txt')
        if not lbl.exists():
            continue
        img = cv2.imread(str(ip))
        if img is None:
            continue
        h, w = img.shape[:2]
        gts = []
        for line in lbl.read_text(encoding='utf-8').splitlines():
            if len(line.strip().split()) != 5:
                continue
            gts.append(yolo_to_xyxy(line, w, h))
        if not gts:
            continue
        pred = model.predict(source=str(ip), imgsz=640, conf=conf_thr, verbose=False)[0]
        if pred.boxes is None or len(pred.boxes) == 0:
            preds = []
        else:
            cls  = pred.boxes.cls.cpu().numpy().astype(int)
            xyxy = pred.boxes.xyxy.cpu().numpy()
            preds = list(zip(cls.tolist(), xyxy.tolist()))
        # Greedy GT->prediction matching at IoU >= iou_thr (same class only).
        matched = [False] * len(preds)
        fn = 0
        for gc, gb in gts:
            best = (-1, 0)
            for j, (pc, pb) in enumerate(preds):
                if matched[j] or pc != gc:
                    continue
                iou = iou_xyxy(gb, pb)
                if iou > best[1]:
                    best = (j, iou)
            if best[0] >= 0 and best[1] >= iou_thr:
                matched[best[0]] = True
            else:
                fn += 1
        fp = matched.count(False)
        if fn + fp > 0:
            items.append((ip, fn, fp, gts, preds))
    items.sort(key=lambda t: -(t[1] + t[2]))
    return items[:max_cases]

failures = collect_failures(baseline_model, CLASSES, max_cases=6)
for (ip, fn, fp, gts, preds) in failures:
    img = cv2.imread(str(ip))
    gt_canvas = img.copy()
    for c, b in gts:
        x1, y1, x2, y2 = [int(v) for v in b]
        cv2.rectangle(gt_canvas, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(gt_canvas, CLASSES[c], (x1, max(15, y1 - 3)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    b_pred = baseline_model.predict(source=str(ip), imgsz=640, conf=0.25, verbose=False)[0]
    p_pred = p2_best_model.predict (source=str(ip), imgsz=640, conf=0.25, verbose=False)[0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(gt_canvas, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'GT (FN={fn}, FP={fp})'); axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(draw_predictions(img, b_pred, (0, 165, 255)), cv2.COLOR_BGR2RGB))
    axes[1].set_title('Baseline'); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(draw_predictions(img, p_pred, (255, 80, 80)), cv2.COLOR_BGR2RGB))
    axes[2].set_title('P2'); axes[2].axis('off')
    fig.suptitle(ip.name)
    fig.tight_layout()
    fig.savefig(REPORT_FAILURES / f'{ip.stem}_failure.png', dpi=120)
    plt.close(fig)
print('Failure gallery saved to:', REPORT_FAILURES)

In [ ]:
abl_csv = DRIVE_ABL_DIR / 'ablation_results.csv'
if abl_csv.exists():
    df_abl_full = pd.read_csv(abl_csv)
    needed = {'baseline_test_clean', 'baseline_test_clean_tta',
              'p2_test_clean',       'p2_test_clean_tta'}
    if needed.issubset(set(df_abl_full['exp'].unique())):
        def gv(tag, col='map50'):
            s = df_abl_full[df_abl_full['exp'] == tag]
            return float(s[col].iloc[0]) if len(s) else float('nan')

        hr_tags = [t for t in df_abl_full['exp'].unique()
                   if t.endswith('_imgsz960') or t.endswith('_imgsz1024')]
        baseline_hr = next((t for t in hr_tags if t.startswith('baseline')), None)
        p2_hr       = next((t for t in hr_tags if t.startswith('p2')),       None)

        settings = ['clean 640', 'clean 640 + TTA']
        b_vals = [gv('baseline_test_clean'), gv('baseline_test_clean_tta')]
        p_vals = [gv('p2_test_clean'),       gv('p2_test_clean_tta')]
        if baseline_hr and p2_hr:
            settings.append('clean 960')
            b_vals.append(gv(baseline_hr))
            p_vals.append(gv(p2_hr))

        x = np.arange(len(settings)); w = 0.38
        fig, ax = plt.subplots(figsize=(8.5, 5))
        ax.bar(x - w/2, b_vals, w, label='Baseline')
        ax.bar(x + w/2, p_vals, w, label='P2')
        ax.set_xticks(x); ax.set_xticklabels(settings)
        ax.set_ylabel('mAP@0.5'); ax.set_ylim(0, 1.0)
        ax.set_title('Inference-time enhancements: TTA + high-resolution')
        ax.grid(axis='y', alpha=0.3); ax.legend()
        for xi, v in zip(x - w/2, b_vals):
            if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
        for xi, v in zip(x + w/2, p_vals):
            if v == v: ax.text(xi, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
        fig.tight_layout()
        fig.savefig(REPORT_PLOTS / 'tta_highres_compare.png', dpi=150)
        plt.show()
    else:
        print('TTA / high-res rows missing from ablation_results.csv; skipping that plot.')
else:
    print('ablation_results.csv not found.')

illum_csv = REPO_ROOT / 'illumination_outputs' / 'illumination_results.csv'
if illum_csv.exists():
    df_illum = pd.read_csv(illum_csv)
    lights   = sorted(df_illum['light'].unique())
    x        = np.arange(len(lights)); w = 0.38

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, title, ylim in [
        (axes[0], 'map50', 'mAP@0.5 per illumination',           (0.0, 1.0)),
        (axes[1], 'fnr',   'False Negative Rate per illumination', (0.0, 0.5)),
    ]:
        b = [float(df_illum[(df_illum['model'] == 'baseline') & (df_illum['light'] == L)][col].iloc[0]) for L in lights]
        p = [float(df_illum[(df_illum['model'] == 'p2')       & (df_illum['light'] == L)][col].iloc[0]) for L in lights]
        ax.bar(x - w/2, b, w, label='Baseline')
        ax.bar(x + w/2, p, w, label='P2')
        ax.set_xticks(x); ax.set_xticklabels(lights, rotation=20)
        ax.set_title(title); ax.set_ylim(*ylim)
        ax.grid(axis='y', alpha=0.3); ax.legend()
    fig.suptitle('Multi-Illumination Robustness on Test Split')
    fig.tight_layout()
    fig.savefig(REPORT_PLOTS / 'illumination_robustness.png', dpi=150)
    plt.show()

    df_illum.to_csv(REPORT_DIR / 'illumination_results.csv', index=False)
else:
    print('illumination_outputs/illumination_results.csv not found - skipping illumination plot.')

In [ ]:
summary = {
    'classes'                 : CLASSES,
    'dataset'                 : stats_df.to_dict(orient='records'),
    'small_defect_ratio_train': small_ratio,
}
if df_abl is not None:
    summary['ablation']  = df_abl.where(df_abl.notna(), None).to_dict(orient='records')
if pc_table is not None:
    summary['per_class'] = pc_table.where(pc_table.notna(), None).to_dict(orient='records')
summary_path = REPORT_DIR / 'report_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, default=str, allow_nan=False), encoding='utf-8')

if df_abl is not None:
    keep = ['baseline_test_clean',     'p2_test_clean',
            'baseline_test_noise_s10', 'p2_test_noise_s10',
            'baseline_test_noise_s20', 'p2_test_noise_s20',
            'baseline_test_noise_s30', 'p2_test_noise_s30']
    sub = df_abl[df_abl['exp'].isin(keep)].sort_values('exp')
    cols = ['exp', 'map50', 'map50_95', 'precision', 'recall', 'f1', 'fnr', 'fps']
    (REPORT_DIR / 'ablation_table.tex').write_text(
        sub[cols].to_latex(index=False, float_format='%.4f',
                           caption='Baseline vs P2 metrics on clean and noisy test sets.',
                           label='tab:ablation'),
        encoding='utf-8',
    )
print('Summary:', summary_path)

In [ ]:
for p in sorted(REPORT_DIR.rglob('*')):
    if p.is_file():
        print(p.relative_to(REPORT_DIR))